In [1]:
from Bio import AlignIO, SeqIO
from Bio.Seq import Seq
import pandas as pd
from Bio.SeqRecord import SeqRecord
import re


In [2]:
def cross_check(fasta_path, region):
    SPIKE_COORDS = (21563, 25321)
    RBD_COORDS = (22553, 23155)
    (start, end) = SPIKE_COORDS if region == "spike" else RBD_COORDS

    rec = SeqIO.read(fasta_path, format="fasta")

    wt = []
    aa_pos_ls = []
    coords = []
    aa_pos = 1 if region == "spike" else 331 # 1-based idx

    for pos in range(start-1, end, 3): # 0-based idx
        try:
            ref_seq = str(rec.seq)
            aa = str(Seq(ref_seq[pos: pos+3]).translate())

            if aa  == "*" or aa == "-":
                aa_pos += 1
                continue
            wt.append(aa)
            aa_pos_ls.append(aa_pos)
            coords.append((int(pos), int(pos+3))) # 0-based idx
            aa_pos += 1
        except:
            aa_pos += 1
            continue
        
    df = pd.DataFrame({'wildtype': wt, 'site': aa_pos_ls, "coords": coords}) if region == "spike" else pd.DataFrame({'wt': wt, 'pos': aa_pos_ls, "coords": coords})

    return df


In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seqs_aligned_to_Hu-1.fa.split/gene_seqs_aligned_to_Hu-1.part_NC_045512.2_XBB.1.5_spike.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/XBB.1.5/spike/XBB.1.5_spike_DMS.csv"

df = cross_check(fasta_path, region="spike")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["site"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["site", "wildtype"], indicator=True)
m = m[m["_merge"] != "both"]
m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]] if pd.notna(x) else "---")
m

,wildtype,site,coords,mutant,human sera escape,spike mediated entry,ACE2 binding,sequential_site,region,_merge,codon
139,V,143,NaN,-,-0.03895,-0.08258,-0.3188,140.0,NTD,right_only,NaN
417,N,422,"(22825, 22828)",NaN,NaN,NaN,NaN,NaN,NaN,left_only,aac
475,C,480,"(22999, 23002)",NaN,NaN,NaN,NaN,NaN,NaN,left_only,tgt
501,Q,506,"(23077, 23080)",NaN,NaN,NaN,NaN,NaN,NaN,left_only,caa
526,T,531,"(23152, 23155)",NaN,NaN,NaN,NaN,NaN,NaN,left_only,acc
558,Q,563,"(23248, 23251)",NaN,NaN,NaN,NaN,NaN,NaN,left_only,caa
574,P,579,"(23296, 23299)",NaN,NaN,NaN,NaN,NaN,NaN,left_only,cca
715,I,720,"(23719, 23722)",NaN,NaN,NaN,NaN,NaN,NaN,left_only,atc
720,E,725,"(23734, 23737)",NaN,NaN,NaN,NaN,NaN,NaN,left_only,gag
744,C,749,"(23806, 23809)",NaN,NaN,NaN,NaN,NaN,NaN,left_only,tgt


In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seq_aligned_to_Hu-1.part_NC_045512.2_BA.2_spike.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/BA.2/spike/BA.2_spike_summary.csv"

df = cross_check(fasta_path, region="spike")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["site"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["site", "wildtype"], indicator=True)
m = m[m["_merge"] != "both"]
m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]])
m

,wildtype,site,coords,mutant,spike mediated entry,ACE2 binding,sequential_site,region,_merge,codon
418,N,422,"(22825, 22828)",NaN,NaN,NaN,NaN,NaN,left_only,aac
433,N,437,"(22870, 22873)",NaN,NaN,NaN,NaN,NaN,left_only,aac
520,V,524,"(23131, 23134)",NaN,NaN,NaN,NaN,NaN,left_only,gtg
559,Q,563,"(23248, 23251)",NaN,NaN,NaN,NaN,NaN,left_only,caa
716,I,720,"(23719, 23722)",NaN,NaN,NaN,NaN,NaN,left_only,atc
812,S,816,"(24007, 24010)",NaN,NaN,NaN,NaN,NaN,left_only,tcc
814,I,818,"(24013, 24016)",NaN,NaN,NaN,NaN,NaN,left_only,att
970,S,974,"(24481, 24484)",NaN,NaN,NaN,NaN,NaN,left_only,tcc


In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seq_aligned_to_Hu-1.part_NC_045512.2_BA.1_rbd.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/BA.1/rbd/mut1_BA.1_Omicron_baseline_EPI_ISL_10000028.csv"

df = cross_check(fasta_path, region="rbd")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["pos"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["pos", "wt"], indicator=True)
m = m[m["_merge"] != "both"]
m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]])
m

,wt,pos,coords,mut1,num_aa,_merge,codon


In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seq_aligned_to_Hu-1.part_NC_045512.2_BA.2_rbd.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/BA.2/rbd/mut1_BA.2_Omicron_baseline_EPI_ISL_10000005.csv"

df = cross_check(fasta_path, region="rbd")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["pos"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["pos", "wt"], indicator=True)
m = m[m["_merge"] != "both"]
m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]])
m

,wt,pos,coords,mut1,num_aa,_merge,codon


In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seq_aligned_to_Hu-1.part_NC_045512.2_BA.2.75_rbd.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/BA.2.75/rbd/mut1_BA.2.75_EPI_ISL_13302209.csv"

df = cross_check(fasta_path, region="rbd")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["pos"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["pos", "wt"], indicator=True)
m = m[m["_merge"] != "both"]
m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]])
m

,wt,pos,coords,mut1,num_aa,_merge,codon


In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seq_aligned_to_Hu-1.part_NC_045512.2_BA.4_BA.5_rbd.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/BA.4_BA.5/rbd/mut1_BA.4_BA.5_EPI_ISL_11207535.csv"

df = cross_check(fasta_path, region="rbd")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["pos"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["pos", "wt"], indicator=True)
m = m[m["_merge"] != "both"]
m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]])
m

,wt,pos,coords,mut1,num_aa,_merge,codon


JN.1, KP.2, and KP.3 have some mismatches with the mut file at aa_pos 482 or 483 since there's gaps or N in the region

In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seq_aligned_to_Hu-1.part_NC_045512.2_JN.1_rbd.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/JN.1/rbd/mut1_JN_1_EPI_ISL_18373905.csv"

df = cross_check(fasta_path, region="rbd")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["pos"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["pos", "wt"], indicator=True)
m = m[m["_merge"] != "both"]
# m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]])
m

,wt,pos,coords,mut1,num_aa,_merge
151,G,482,NaN,ACDRSV,6,right_only


In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seq_aligned_to_Hu-1.part_NC_045512.2_KP.2_rbd.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/KP.2/rbd/mut1_KP_2_EPI_ISL_18916251_del.csv"

df = cross_check(fasta_path, region="rbd")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["pos"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["pos", "wt"], indicator=True)
m = m[m["_merge"] != "both"]
m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]])
m

,wt,pos,coords,mut1,num_aa,_merge,codon
152,X,483,"(23008, 23011)",NaN,NaN,left_only,nnt


In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seq_aligned_to_Hu-1.part_NC_045512.2_KP.3_rbd.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/KP.3/rbd/mut1_KP_3_EPI_ISL_19036766.csv"

df = cross_check(fasta_path, region="rbd")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["pos"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["pos", "wt"], indicator=True)
m = m[m["_merge"] != "both"]
# m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]])
m

,wt,pos,coords,mut1,num_aa,_merge
151,G,482,NaN,ACDRSV,6,right_only


In [ ]:
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seq_aligned_to_Hu-1.part_NC_045512.2_XBB.1.5_rbd.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/XBB.1.5/rbd/mut1_XBB_1_5_EPI_ISL_17054053.csv"

df = cross_check(fasta_path, region="rbd")
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["pos"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", on=["pos", "wt"], indicator=True)
m = m[m["_merge"] != "both"]
# m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]])
m

,wt,pos,coords,mut1,num_aa,_merge
159,F,490,"(23029, 23032)",NaN,NaN,left_only
160,S,490,NaN,ACFPTY,6.0,right_only


In [16]:
# we don't have the mut file for KP.3_spike here, so use the rbd file instead to check the rbd region..
fasta_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/hu-1_all_kp.3.fa.split/hu-1_all_kp.3.part_NC_045512.2_KP.3_spike.fa"
csv_path = "/home/yutianc/sars-cov-2-dms-ref/data/KP.3/rbd/mut1_KP_3_EPI_ISL_19036766.csv"

df = cross_check(fasta_path, region="spike")
# convert site to str for merging, the mut file has 16a, 16b, ... for site
# df["site"] = df["site"].astype(str)
df_true = pd.read_csv(csv_path)
df_true = df_true.drop_duplicates(subset=["pos", "wt"])
ref_seq = str(SeqIO.read(fasta_path, format="fasta").seq)

m = pd.merge(df, df_true, how="outer", left_on=["site", "wildtype"], right_on=["pos", "wt"], indicator=True)
m = m[(m["_merge"] != "both") & (df["site"]>=331) & (df["site"] <=531)]
m["codon"] = m["coords"].apply(lambda x: ref_seq[x[0]: x[1]] if pd.notna(x) else "---")

m

,wildtype,site,coords,wt,pos,mut1,num_aa,_merge,codon


In [55]:
# for the ref with .gb file, get the gene seq from gb file; otherwise, extract the seq from alignment file
SPIKE_COORDS = [21563, 25384] 
RBD_COORDS = [22553, 23155]

SPIKE_RE = r"(?i)(?:^|[^A-Za-z0-9])spike(?:$|[^A-Za-z0-9])"
RBD_RE   = r"(?i)(?:^|[^A-Za-z0-9])rbd(?:$|[^A-Za-z0-9])"

REF_PATH = "/home/yutianc/sars-cov-2-dms-ref/data/Hu-1/NC_045512.2.fasta"

id_mapping = {"EPI_ISL_10000028": "NC_045512.2_BA.1", 
              "EPI_ISL_10000005": "NC_045512.2_BA.2", 
              "EPI_ISL_13302209": "NC_045512.2_BA.2.75", 
              "EPI_ISL_11207535": "NC_045512.2_BA.4_BA.5",
              "EPI_ISL_18373905": "NC_045512.2_JN.1", 
              "EPI_ISL_18916251": "NC_045512.2_KP.2",
              "EPI_ISL_19036766": "NC_045512.2_KP.3",
              "EPI_ISL_17054053": "NC_045512.2_XBB.1.5"
              }


def get_gene_from_gb_file(path_to_gb_file, out_name, out_path, apply_ref=False):
    if re.search(SPIKE_RE, out_name):
        region_coords = SPIKE_COORDS
    elif re.search(RBD_RE, out_name):
        region_coords = RBD_COORDS

    dms_ref = SeqIO.read(path_to_gb_file, format="genbank")
    (start, end) = next((i.location.start, i.location.end) for i in dms_ref.features if i.type == "gene")
    gene_seq = dms_ref.seq[start: end]

    if apply_ref:
        ref = SeqIO.read(REF_PATH, format="fasta")
        seq = str(ref.seq)[:region_coords[0]-1] + gene_seq + str(ref.seq)[region_coords[0]-1+len(gene_seq):]
        assert len(seq) == 29903
    else:
        seq = gene_seq

    gene_rec = SeqRecord(Seq(seq), id=out_name, name=out_name, description=out_name)
    with open(out_path, "a") as handle:   
        SeqIO.write(gene_rec, handle, "fasta")


def get_gene_from_msa_algn_file(path_to_algn_file, out_path):
    algn = AlignIO.read(path_to_algn_file, format="fasta") 
    rbd_algn = algn[1:, 22553-1: 23155] # exclude Hu-1, 0-based idx

    for algn in rbd_algn:
        gene_seq = str(algn.seq)
        label = id_mapping[algn.id.split("|")[1]] + "_rbd"
        gene_rec = SeqRecord(Seq(gene_seq), id=label, name=label, description=label)

        with open(out_path, "a") as handle:   
            SeqIO.write(gene_rec, handle, "fasta")


def apply_dms_ref_from_msa_algn_file(path_to_algn_file, out_path, apply_ref=False):
    algn = AlignIO.read(path_to_algn_file, format="fasta") 
    algn = algn[1:, :]
    # rbd_algn = algn[1:, 22553-1: 23155] # exclude Hu-1, 0-based idx
    # spike_algn = algn[1:, 21563-1: 25321]

    for a in algn:
        if re.search(SPIKE_RE, a.id):
            region_coords = SPIKE_COORDS
            gene_seq = str(a.seq[21563-1: 25321])
        elif re.search(RBD_RE, a.id):
            region_coords = RBD_COORDS
            gene_seq = str(a.seq[22553-1: 23155])

        if apply_ref:
            ref = SeqIO.read(REF_PATH, format="fasta")
            seq = str(ref.seq)[:region_coords[0]-1] + gene_seq + str(ref.seq)[region_coords[0]-1+len(gene_seq):]
            assert len(seq) == 29903
        else:
            seq = gene_seq
        
        gene_rec = SeqRecord(Seq(seq), id=a.id, name=a.id, description=a.id)

        with open(out_path, "a") as handle:   
            SeqIO.write(gene_rec, handle, "fasta")


In [ ]:
# extract the gene seq
out_path = "/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seqs.fa"

get_gene_from_gb_file("/home/yutianc/sars-cov-2-dms-ref/data/BA.1/rbd/PacBio_amplicon_BA1.gb", "NC_045512.2_BA.1_rbd", out_path)
get_gene_from_gb_file("/home/yutianc/sars-cov-2-dms-ref/data/BA.2/rbd/PacBio_amplicon_BA2.gb", "NC_045512.2_BA.2_rbd", out_path)
get_gene_from_gb_file("/home/yutianc/sars-cov-2-dms-ref/data/BA.2/spike/PacBio_amplicon.gb", "NC_045512.2_BA.2_spike", out_path)
get_gene_from_gb_file("/home/yutianc/sars-cov-2-dms-ref/data/XBB.1.5/rbd/PacBio_amplicon.gb", "NC_045512.2_XBB.1.5_rbd", out_path)
get_gene_from_gb_file("/home/yutianc/sars-cov-2-dms-ref/data/XBB.1.5/spike/PacBio_amplicon.gb", "NC_045512.2_XBB.1.5_spike", out_path)
get_gene_from_gb_file("/home/yutianc/sars-cov-2-dms-ref/data/KP.3/spike/PacBio_amplicon.gb", "NC_045512.2_KP.3_spike", out_path)


# for the those who do not have .gb file, align them to Hu-1 to get the gene seq for RBD
get_gene_from_msa_algn_file("/home/yutianc/sars-cov-2-dms-ref/data/Hu-1_BA.2.75_BA.4_BA.5_JN.1_KP.2_KP.3.fa", out_path)

Run Mafft after this to align gene seq to Hu-1:

mafft --auto --keeplength --addfragments gene_seqs.fa NC_045512.2.fasta | seqkit seq -w 0 > gene_seq_aligned_to_Hu-1.fa

In [59]:
# apply Hu-1 to gene seq to get the full seqs 
out_path = "/home/yutianc/sars-cov-2-dms-ref/output/hu1_with_dms_ref/all.fa"
apply_dms_ref_from_msa_algn_file("/home/yutianc/sars-cov-2-dms-ref/output/dms_gene_seq/gene_seqs_aligned_to_Hu-1.fa", out_path, True)
